In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_path = "/content/drive/MyDrive/AI-Powered Hospitality Revenue Optimization/Dataset/"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv(base_path + "hotel_bookings_cleaned.csv")

In [5]:
df.shape

(87230, 39)

In [6]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_stay_nights,total_guests,estimated_revenue,adr_invalid,invalid_guest_count,adr_outlier
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,Check-Out,01-07-15,2015-07-01,0,2.0,0.0,False,False,False
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,Check-Out,01-07-15,2015-07-01,0,2.0,0.0,False,False,False
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,Check-Out,02-07-15,2015-07-01,1,1.0,75.0,False,False,False
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,Check-Out,02-07-15,2015-07-01,1,1.0,75.0,False,False,False
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,1,Check-Out,03-07-15,2015-07-01,2,2.0,196.0,False,False,False


In [7]:
daily_demand = (
    df.groupby(["arrival_date", "hotel"])
      .agg(
          bookings=("hotel", "size"),
          cancellations=("is_canceled", "sum"),
          avg_adr=("adr", "mean"),
          total_revenue=("estimated_revenue", "sum"),
          avg_lead_time=("lead_time", "mean"),
          avg_stay_nights=("total_stay_nights", "mean")
      )
      .reset_index()
)

daily_demand["cancellation_rate"] = (
    daily_demand["cancellations"] /
    daily_demand["bookings"] * 100
)

daily_demand.head()

,arrival_date,hotel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,cancellation_rate
0,2015-07-01,City Hotel,12,6,78.875000,3281.50,166.916667,3.416667,50.000000
1,2015-07-01,Resort Hotel,41,5,90.242927,17663.24,86.609756,4.804878,12.195122
2,2015-07-02,City Hotel,11,10,70.224545,2657.61,142.181818,3.363636,90.909091
3,2015-07-02,Resort Hotel,43,9,101.161395,23706.00,69.744186,5.697674,20.930233
4,2015-07-03,City Hotel,11,6,68.501818,1884.89,49.272727,2.545455,54.545455


In [8]:
print("Rows:", len(daily_demand))
print("Columns:", daily_demand.columns.tolist())

Rows: 1586
Columns: ['arrival_date', 'hotel', 'bookings', 'cancellations', 'avg_adr', 'total_revenue', 'avg_lead_time', 'avg_stay_nights', 'cancellation_rate']


In [9]:
monthly_performance = (
    df.groupby([
        "arrival_date_year",
        "arrival_date_month",
        "hotel"
    ])
    .agg(
        bookings=("hotel", "size"),
        cancellations=("is_canceled", "sum"),
        avg_adr=("adr", "mean"),
        total_revenue=("estimated_revenue", "sum"),
        avg_lead_time=("lead_time", "mean"),
        avg_stay_nights=("total_stay_nights", "mean")
    )
    .reset_index()
)

In [10]:
month_map = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

monthly_performance["month_number"] = (
    monthly_performance["arrival_date_month"].map(month_map)
)

monthly_performance["cancellation_rate"] = (
    monthly_performance["cancellations"] /
    monthly_performance["bookings"] * 100
)

monthly_performance = monthly_performance.sort_values(
    ["arrival_date_year", "month_number", "hotel"]
)

monthly_performance.head()

,arrival_date_year,arrival_date_month,hotel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,month_number,cancellation_rate
4,2015,July,City Hotel,393,232,67.153028,93074.80,114.139949,3.470738,7,59.033079
5,2015,July,Resort Hotel,1279,280,126.478045,833412.28,69.184519,5.124316,7,21.892103
0,2015,August,City Hotel,1101,229,83.433170,273750.45,40.858311,2.980018,8,20.799273
1,2015,August,Resort Hotel,1346,342,155.902912,1132045.82,73.782318,5.447251,8,25.408618
10,2015,September,City Hotel,1667,313,107.116053,520787.11,47.787642,2.847630,9,18.776245


In [11]:
market_segment = (
    df.groupby(["hotel", "market_segment"])
      .agg(
          bookings=("hotel", "size"),
          cancellations=("is_canceled", "sum"),
          avg_adr=("adr", "mean"),
          total_revenue=("estimated_revenue", "sum"),
          avg_lead_time=("lead_time", "mean"),
          avg_stay_nights=("total_stay_nights", "mean")
      )
      .reset_index()
)

market_segment["cancellation_rate"] = (
    market_segment["cancellations"] /
    market_segment["bookings"] * 100
)

market_segment

,hotel,market_segment,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,cancellation_rate
0,City Hotel,Aviation,226,45,100.613628,83033.36,4.296460,3.588496,19.911504
1,City Hotel,Complementary,503,54,2.802048,2937.99,11.395626,1.497018,10.735586
2,City Hotel,Corporate,2218,263,83.020234,359977.16,18.348963,1.876465,11.857529
3,City Hotel,Direct,5538,912,121.243682,1959007.34,49.367642,2.871975,16.468039
4,City Hotel,Groups,2619,887,85.262047,597717.48,133.986636,2.627721,33.867889
5,City Hotel,Offline TA/TO,7239,1257,87.632267,2029059.92,97.089791,3.229728,17.364277
6,City Hotel,Online TA,34929,12615,119.971001,13733252.11,79.298348,3.306364,36.116121
7,City Hotel,Undefined,2,2,15.000000,48.00,1.500000,1.500000,100.000000
8,Resort Hotel,Complementary,189,31,3.868466,2136.53,20.285714,2.185185,16.402116
9,Resort Hotel,Corporate,1982,246,51.920873,234166.92,13.906155,2.181130,12.411705


In [12]:
channel_performance = (
    df.groupby(["hotel", "distribution_channel"])
      .agg(
          bookings=("hotel", "size"),
          cancellations=("is_canceled", "sum"),
          avg_adr=("adr", "mean"),
          total_revenue=("estimated_revenue", "sum"),
          avg_lead_time=("lead_time", "mean")
      )
      .reset_index()
)

channel_performance["cancellation_rate"] = (
    channel_performance["cancellations"] /
    channel_performance["bookings"] * 100
)

channel_performance

,hotel,distribution_channel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,cancellation_rate
0,City Hotel,Corporate,2591,330,83.777368,476593.31,25.321883,12.736395
1,City Hotel,Direct,6056,971,112.606688,1977560.93,46.350396,16.033686
2,City Hotel,GDS,181,36,120.317845,43628.82,20.121547,19.889503
3,City Hotel,TA/TO,44442,14694,112.663552,16266642.80,85.378628,33.063318
4,City Hotel,Undefined,4,4,29.625000,607.50,3.000000,100.000000
5,Resort Hotel,Corporate,2471,316,53.036835,357531.51,41.984622,12.788345
6,Resort Hotel,Direct,6898,952,106.567140,3021469.21,57.796028,13.801102
7,Resort Hotel,TA/TO,24586,6706,101.578317,12307057.35,94.728301,27.275685
8,Resort Hotel,Undefined,1,0,112.700000,563.50,103.000000,0.000000


In [13]:
print("Daily Demand:", daily_demand.shape)
print("Monthly Performance:", monthly_performance.shape)
print("Market Segment:", market_segment.shape)
print("Channel Performance:", channel_performance.shape)

Daily Demand: (1586, 9)
Monthly Performance: (52, 11)
Market Segment: (14, 9)
Channel Performance: (9, 8)


In [14]:
daily_demand.head()

,arrival_date,hotel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,cancellation_rate
0,2015-07-01,City Hotel,12,6,78.875000,3281.50,166.916667,3.416667,50.000000
1,2015-07-01,Resort Hotel,41,5,90.242927,17663.24,86.609756,4.804878,12.195122
2,2015-07-02,City Hotel,11,10,70.224545,2657.61,142.181818,3.363636,90.909091
3,2015-07-02,Resort Hotel,43,9,101.161395,23706.00,69.744186,5.697674,20.930233
4,2015-07-03,City Hotel,11,6,68.501818,1884.89,49.272727,2.545455,54.545455


In [15]:
monthly_performance.head()

,arrival_date_year,arrival_date_month,hotel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,month_number,cancellation_rate
4,2015,July,City Hotel,393,232,67.153028,93074.80,114.139949,3.470738,7,59.033079
5,2015,July,Resort Hotel,1279,280,126.478045,833412.28,69.184519,5.124316,7,21.892103
0,2015,August,City Hotel,1101,229,83.433170,273750.45,40.858311,2.980018,8,20.799273
1,2015,August,Resort Hotel,1346,342,155.902912,1132045.82,73.782318,5.447251,8,25.408618
10,2015,September,City Hotel,1667,313,107.116053,520787.11,47.787642,2.847630,9,18.776245


In [16]:
market_segment.head()

,hotel,market_segment,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,avg_stay_nights,cancellation_rate
0,City Hotel,Aviation,226,45,100.613628,83033.36,4.296460,3.588496,19.911504
1,City Hotel,Complementary,503,54,2.802048,2937.99,11.395626,1.497018,10.735586
2,City Hotel,Corporate,2218,263,83.020234,359977.16,18.348963,1.876465,11.857529
3,City Hotel,Direct,5538,912,121.243682,1959007.34,49.367642,2.871975,16.468039
4,City Hotel,Groups,2619,887,85.262047,597717.48,133.986636,2.627721,33.867889


In [17]:
channel_performance.head()

,hotel,distribution_channel,bookings,cancellations,avg_adr,total_revenue,avg_lead_time,cancellation_rate
0,City Hotel,Corporate,2591,330,83.777368,476593.31,25.321883,12.736395
1,City Hotel,Direct,6056,971,112.606688,1977560.93,46.350396,16.033686
2,City Hotel,GDS,181,36,120.317845,43628.82,20.121547,19.889503
3,City Hotel,TA/TO,44442,14694,112.663552,16266642.80,85.378628,33.063318
4,City Hotel,Undefined,4,4,29.625000,607.50,3.000000,100.000000


In [18]:
new_path = "/content/drive/MyDrive/AI-Powered Hospitality Revenue Optimization/Dataset/Normalized Data for SQL/"

In [19]:
daily_demand.to_csv(new_path + "daily_demand.csv", index=False)
monthly_performance.to_csv(new_path + "monthly_performance.csv", index=False)
market_segment.to_csv(new_path + "market_segment.csv", index=False)
channel_performance.to_csv(new_path + "channel_performance.csv", index=False)